# Synthetic Data Generation on the German Credit Risk Dataset

## Introduction

In this notebook we focus on the generation of synthetic data by leveraging the models trained in the previous steps. Each trained model is loaded from its saved state and used to produce synthetic samples of the German Credit Risk dataset.

These generated datasets will then serve as the foundation for the upcoming evaluation phase.

## 0 Imports and function definition

In this notebook we import the essential libraries required for loading the trained models and generating synthetic data. These include pandas for data handling, pickle for loading saved models, and the corresponding packages for each generative approach (GReaT and SDV synthesizers).

### Downloads

In [1]:
!pip install be_great peft==0.9.0 sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Imports

In [2]:
import pandas as pd
import os
from be_great import GReaT
from sdv.single_table import TVAESynthesizer, CTGANSynthesizer, CopulaGANSynthesizer
from sdv.metadata import SingleTableMetadata
import pickle

In [3]:
os.environ["WANDB_DISABLED"] = "true" #to avoid the requests from WANDB

### Functions definitions

The function load_model is a simple utility that loads a previously saved generative model from disk using pickle. By providing the file path, it restores the model object into memory, making it ready for use in data generation.

In [4]:
def load_model(file_path):
  with(open(file_path, 'rb')) as f:
    model = pickle.load(f)
  return model

These functions define a unified interface for generating synthetic data with the trained models. Each function takes as input a generative model and the desired number of rows, then calls the model’s sampling method to produce a synthetic dataset. While the SDV-based models (CopulaGAN, TVAE, CTGAN) use sample(num_rows=...), the GReaT-based models rely on sample(n_samples=..., max_length=...) to account for text-based generation constraints.

In [5]:
def create_synthetic_data_CopulaGAN(model, num_rows):
    synthetic_data = model.sample(num_rows=num_rows)
    return synthetic_data

In [6]:
def create_synthetic_data_TVAE(model, num_rows):
    synthetic_data = model.sample(num_rows=num_rows)
    return synthetic_data

In [7]:
def create_synthetic_data_CTGAN(model, num_rows):
    synthetic_data = model.sample(num_rows=num_rows)
    return synthetic_data

In [8]:
def create_synthetic_data_distil_GReaT(model, num_rows):
    synthetic_data = model.sample(n_samples=num_rows, max_length=2000, guided_sampling=True)
    return synthetic_data

In [9]:
def create_synthetic_data_GReaT(model, num_rows):
    synthetic_data = model.sample(n_samples=num_rows, max_length=2000, guided_sampling=True)
    return synthetic_data

## 1 Sampling

We now move to the sampling phase, where synthetic records are generated from the trained models. To ensure a fair comparison in the upcoming evaluations, we fix the number of generated rows to 1,000, which matches the size of the original German Credit Risk dataset.

For each model, the workflow follows the same steps:

Load the pre-trained model from file.

Generate synthetic data with the specified number of rows.

Display a preview of the generated dataset.

Save the synthetic records into a CSV file for later use.

This process is applied consistently across all models (CopulaGAN, TVAE, CTGAN, GReaT, and distil-GReaT).

In [10]:
num_rows = 1000

### CopulaGAN

In [11]:
model_CopulaGAN = load_model('copulagan_german.pkl')

In [12]:
synthetic_data_CopulaGAN = create_synthetic_data_CopulaGAN(model_CopulaGAN, num_rows)

In [13]:
synthetic_data_CopulaGAN.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,28.0,female,2,rent,little,rich,6135.0,56.0,furniture/equipment,good
1,59.0,male,3,own,little,Unknown,1740.0,11.0,radio/TV,bad
2,22.0,male,2,own,little,little,2156.0,27.0,car,good
3,25.0,male,2,own,little,moderate,1215.0,17.0,furniture/equipment,good
4,49.0,male,2,own,little,moderate,10546.0,14.0,furniture/equipment,bad


In [14]:
synthetic_data_CopulaGAN.to_csv('synthetic_data_german_CopulaGAN.csv', index=False)

### TVAE

In [15]:
model_TVAE = load_model('tvae_german.pkl')

In [16]:
synthetic_data_TVAE = create_synthetic_data_TVAE(model_TVAE, num_rows)

In [17]:
synthetic_data_TVAE.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,24.0,male,2,own,little,Unknown,1683.0,13.0,radio/TV,good
1,27.0,female,2,own,little,little,1157.0,11.0,furniture/equipment,bad
2,22.0,male,2,own,little,Unknown,5067.0,24.0,radio/TV,good
3,25.0,male,2,own,little,little,5978.0,39.0,car,bad
4,29.0,male,2,own,little,Unknown,1148.0,6.0,radio/TV,good


In [18]:
synthetic_data_TVAE.to_csv('synthetic_data_german_TVAE.csv', index=False)

### CTGAN

In [19]:
model_CTGAN = load_model('ctgan_german.pkl')

In [20]:
synthetic_data_CTGAN = create_synthetic_data_CTGAN(model_CTGAN, num_rows)

In [21]:
synthetic_data_CTGAN.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,24.0,male,2,own,Unknown,rich,1513.0,4.0,radio/TV,good
1,57.0,male,3,own,little,moderate,3554.0,4.0,radio/TV,good
2,31.0,male,2,own,quite rich,Unknown,1465.0,9.0,business,bad
3,66.0,male,2,rent,little,Unknown,3315.0,18.0,radio/TV,good
4,75.0,male,1,own,little,little,250.0,5.0,radio/TV,good


In [22]:
synthetic_data_CTGAN.to_csv('synthetic_data_german_CTGAN.csv', index=False)

### Distil-GReaT

In [23]:
model_distil_GReaT = load_model('distil_GReaT_german.pkl')

In [24]:
synthetic_data_distil_GReaT = create_synthetic_data_distil_GReaT(model_distil_GReaT, num_rows)

100%|██████████| 1000/1000 [36:35<00:00,  2.20s/it]


In [25]:
synthetic_data_distil_GReaT.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,34.0,male,2.0,own,little,Unknown,966.0,18.0,car,good
1,51.0,male,2.0,ownThe,Unknown,little,618.0,18.0,business,bad
2,41.0,male,2.0,ownWhat,little,little,2630.0,12.0,furniture/equipment,good
3,28.0,male,2.0,own,little,Unknown,942.0,12.0,radio/TV,good
4,26.0,female,3.0,own,littleA group of young men are taking shelter ...,moderate,1579.0,12.0,furniture/equipment,good


In [26]:
synthetic_data_distil_GReaT.to_csv('synthetic_data_german_distil_GReaT.csv', index=False)

GReaT

In [27]:
model_GReaT = load_model('GReaT_german.pkl')

In [28]:
synthetic_data_GReaT = create_synthetic_data_GReaT(model_GReaT, num_rows)

100%|██████████| 1000/1000 [1:02:45<00:00,  3.77s/it]


In [29]:
synthetic_data_GReaT.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,40.0,femaleA new report from the National Institute...,2,own,little,UnknownHow do you use your desktop computer,1282.0,18.0,radio/TV,good
1,33.0,male,2,own,little,Unknown,1282.0,36.0,car,bad
2,62.0,male,2,own,little,littleIf you're shopping for a car,4397.0,18.0,car,good
3,37.0,male,2,free,little,little,2680.0,9.0,radio/TV,bad
4,51.0,female,2,rent,moderateThe following article is from the Dece...,moderate,1178.0,12.0,furniture/equipment,good


In [30]:
synthetic_data_GReaT.to_csv('synthetic_data_german_GReaT.csv', index=False)

During synthetic data generation with the GReaT model (both GPT-2 and distil-GPT2 versions), we observe that some categorical columns occasionally contain values that deviate significantly from the original categories. In several cases, the generated entries include additional text or concatenated phrases that extend beyond the expected category labels. For example, the Sex or Saving accounts columns may contain full sentences appended to the original value.

This behavior is likely inherent to the language-model-based architecture of GReaT. Since the model is trained on sequences of tokenized text rather than discrete categorical labels, it can produce outputs that resemble natural language rather than strictly adhering to the original categories. While this allows for flexible and creative sampling, it can also generate artifacts that do not align perfectly with the expected tabular schema.

Such observations highlight a key difference between LLM-based synthetic data generation and traditional tabular generative models (e.g., CopulaGAN, TVAE, CTGAN), where outputs strictly respect the input categories. Awareness of this phenomenon is important when interpreting or using synthetic datasets produced by GReaT.